In [1]:
import os
os.environ["CALITP_BQ_MAX_BYTES"] = str(800_000_000_000)

import shared_utils
import pandas as pd
import geopandas as gpd

import gcsfs
from calitp_data_analysis import get_fs
from calitp_data_analysis import geography_utils, utils
fs = get_fs()
import re
import google.auth
import os
import gcsfs
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

In [2]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
job_density_prv = pd.read_csv(f"{GCS_FILE_PATH}/Job Density Raw File Private.csv")

In [5]:
job_density_tot = pd.read_csv(f"{GCS_FILE_PATH}/Job Density Raw File Total.csv")

In [6]:
job_density_prv = job_density_prv.rename(columns={'C000': 'jobs_prv'})
job_density_tot = job_density_tot.rename(columns={'C000': 'jobs_tot'})

In [7]:
job_density_prv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368595 entries, 0 to 368594
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   h_geocode  368595 non-null  int64 
 1   segment    368595 non-null  object
 2   year       368595 non-null  int64 
 3   jobs_prv   368595 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 11.2+ MB


In [8]:
job_density_tot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 370418 entries, 0 to 370417
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   h_geocode  370418 non-null  int64 
 1   segment    370418 non-null  object
 2   year       370418 non-null  int64 
 3   jobs_tot   370418 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 11.3+ MB


In [10]:
merged_df = pd.merge(
    job_density_tot,
    job_density_prv[['h_geocode', 'jobs_prv']],
    on='h_geocode',
    how='left'
)

In [11]:
#Sort by GEOID
merged_df = merged_df.sort_values(by='h_geocode')

#Create jobs_fed = jobs_total - jobs_prv
merged_df['jobs_fed'] = merged_df['jobs_tot'] - merged_df['jobs_prv']


In [12]:
merged_df['GEOID'] = merged_df['h_geocode'].astype(str).str.zfill(15).str[:11]

In [13]:
grouped_df = merged_df.groupby('GEOID', as_index=False).agg({
    'jobs_tot': 'sum',
    'jobs_prv': 'sum',
    'jobs_fed': 'sum',
    'year': 'first'
})

In [14]:
def export_gdf(gdf, filename: str):
    
    gdf.to_parquet(f"{filename}.parquet")
    
    fs.put(
        f"{filename}.parquet",
        f"{GCS_FILE_PATH}/{filename}.parquet",
        token = credentials.token
    )
    
    os.remove(f"{filename}.parquet")
    print(f"saved {GCS_FILE_PATH}/{filename}.parquet")
    
    return

In [15]:
export_gdf(grouped_df, "job_density_2023")

saved gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026/job_density_2023.parquet
